# Quick check: can the arm reach imaging-rack slot 0?

Moves the **empty** gripper from where it is now (pointing down, above the racks) to 10 cm above
slot 0 of the imaging rack (8 holes = 400 mm left of the rack with sample 0), optionally lowers it
to 5 mm above the slot, then brings it back the same way.

Before running:
- Swing the camera and its boom out of the way.
- Gripper empty, hand on the e-stop.
- Do not run anything in another notebook that is connected to the arm at the same time.

Run the cells in order, without restarting the kernel in between.

In [ ]:
from robotic_testing.platforms import init_vla_platform

rt = init_vla_platform(exp_name='imaging_quick_check', connect_db=False)
arm = rt.arm

In [ ]:
# ===== Quick check 1/3: EMPTY gripper from where it is now to 10 cm above imaging-rack slot 0 =====
# Before running: camera and boom swung out of the way, gripper empty, hand on the e-stop.
# Route: straight up from where the arm is now -> across, high above the bench -> down to 10 cm above slot 0.
# These moves are not checked against the code's safety zone, so they are slow and need your "yes".

IMAGING_SLOT_0 = [-246.0, 408.1, 190.6, 180.0, 0.0, 90.0]  # 8 holes (400 mm) left of sample 0
SAFE_Z = 500.0                                              # travel height, well above the flask

x, y, z, roll, pitch, yaw = IMAGING_SLOT_0


def at_speed(mm_s):
    """Normal motion settings, with the speed (mm/s) changed."""
    return {**arm.config.pos_settings_dict['default'], 'speed': mm_s}


start = arm.get_cartesian_pos()          # remembered, so cell 3 can bring the arm back here
print('Arm is now at:', [round(v, 1) for v in start])
turn = max(abs(((a - b) + 180) % 360 - 180) for a, b in zip(start[3:], [roll, pitch, yaw]))
if turn > 5 or start[2] < 240:
    raise SystemExit('The gripper must point straight down (like above the racks) and be at least '
                     '5 cm above the racks (z >= 240) before this check. The arm did not move.')

steps = [
    ('straight up from here',          [start[0], start[1], SAFE_Z, roll, pitch, yaw], 50),
    ('across, high above the bench',   [x, y, SAFE_Z, roll, pitch, yaw], 30),
    ('down to 10 cm above slot 0',     [x, y, z + 100, roll, pitch, yaw], 20),
]
print('Plan:')
for name, pose, speed in steps:
    print(f'  {name:30s} -> {[round(v, 1) for v in pose]} at {speed} mm/s')
if input('Camera moved away, gripper empty, hand on the e-stop? Type yes to start: ').strip().lower() != 'yes':
    raise SystemExit('Cancelled, the arm did not move.')

arm.assert_ready()
for name, pose, speed in steps:
    print('->', name)
    arm.move_to_pos(pose, **at_speed(speed))
print('Stopped at:', [round(v, 1) for v in arm.get_cartesian_pos()])
print('Look from the front and from the side: are the fingers right above slot 0 of the imaging rack?')

In [ ]:
# ===== Quick check 2/3 (optional): lower slowly to 5 mm above the slot =====
target = [x, y, z + 5, roll, pitch, yaw]
if input('Fingers right above slot 0, nothing in the way? Type yes to go down slowly: ').strip().lower() == 'yes':
    arm.move_to_pos(target, **at_speed(8))
    print('Now at:', [round(v, 1) for v in arm.get_cartesian_pos()])
    print('If the fingers are off the slot, note which way and roughly how many mm.')
else:
    print('Stayed where it is.')

In [ ]:
# ===== Quick check 3/3: back to where the arm started (above sample 0), same route =====
cur = arm.get_cartesian_pos()
print('Going back: straight up, across, down to the start position.')
arm.move_to_pos([cur[0], cur[1], SAFE_Z, roll, pitch, yaw], **at_speed(20))
arm.move_to_pos([start[0], start[1], SAFE_Z, roll, pitch, yaw], **at_speed(30))
arm.move_to_pos([start[0], start[1], start[2], roll, pitch, yaw], **at_speed(20))
print('Back at:', [round(v, 1) for v in arm.get_cartesian_pos()])